In [1]:
#cell1
# Setup imports

import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from google.colab import drive
from IPython.display import display

In [2]:
#cell2
# Mount Google Drive silently

import io
import contextlib

with contextlib.redirect_stdout(io.StringIO()):
    drive.mount("/content/drive", force_remount=False)

In [3]:
#cell3
# Define file paths, dataset names, and model names

BASE_DIR = Path("/content/drive/MyDrive/final_project/RAG/answers")

ANSWER_FILES = [
    {
        "dataset": "hotpotqa",
        "model": "qwen3.5",
        "file_name": "hotpotqa_qwen3.5_answers.json",
        "file_path": BASE_DIR / "hotpotqa_qwen3.5_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "qwen3.5",
        "file_name": "2wikimultihopqa_qwen3.5_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_qwen3.5_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "gpt-oss-120b",
        "file_name": "hotpotqa_gpt_oss_120b_answers.json",
        "file_path": BASE_DIR / "hotpotqa_gpt_oss_120b_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "gpt-oss-120b",
        "file_name": "2wikimultihopqa_gpt_oss_120b_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_gpt_oss_120b_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "gemma4",
        "file_name": "hotpotqa_gemma4_answers.json",
        "file_path": BASE_DIR / "hotpotqa_gemma4_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "gemma4",
        "file_name": "2wikimultihopqa_gemma4_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_gemma4_answers.json",
    },
]

In [4]:
#cell4
# Normalize text and tokenize answers

def normalize_text(text):
    """
    Basic normalization for token-level comparison.
    Lowercase, remove punctuation, and normalize spaces.
    """
    if text is None:
        text = ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()

    chars = []
    for ch in text:
        if unicodedata.category(ch).startswith("P"):
            chars.append(" ")
        else:
            chars.append(ch)

    text = "".join(chars)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    """
    Convert answer text to a set of tokens.
    This keeps the same set-overlap logic as the previous notebook.
    """
    normalized = normalize_text(text)
    if not normalized:
        return set()
    return set(normalized.split())

In [5]:
#cell5
# Compute Precision, Recall, and F1 for one example

def compute_token_f1(predicted_answer, ground_truth_answer):
    """
    Compute token-level Precision, Recall, and F1.
    """
    pred_tokens = tokenize(predicted_answer)
    gt_tokens = tokenize(ground_truth_answer)

    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0, 0.0, 0.0

    overlap = pred_tokens.intersection(gt_tokens)

    precision = len(overlap) / len(pred_tokens)
    recall = len(overlap) / len(gt_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return precision, recall, f1

In [6]:
#cell6
# Load one JSON file and compute row-level scores

def load_json_file(file_path):
    """
    Load a JSON answer file.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected a list of records in: {file_path}")

    return data


def evaluate_file(dataset_name, model_name, file_name, file_path):
    """
    Evaluate all questions for one dataset-model file.
    """
    data = load_json_file(file_path)
    rows = []

    for idx, item in enumerate(data):
        gt = item.get("gt", "")
        response = item.get("response", "")
        question_type = item.get("type", "unknown")

        precision, recall, f1 = compute_token_f1(
            predicted_answer=response,
            ground_truth_answer=gt
        )

        rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "file_name": file_name,
            "row_index": idx,
            "source_index": item.get("source_index", idx),
            "type": question_type if question_type is not None else "unknown",
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "question": item.get("question", ""),
            "gt": gt,
            "response": response,
        })

    return pd.DataFrame(rows)

In [7]:
#cell7
# Evaluate all files while keeping dataset, model, and file identity separated

def evaluate_all_files(answer_files):
    """
    Evaluate every JSON file and combine row-level results.
    Each row keeps dataset, model, and file_name to prevent mixing results.
    """
    all_dfs = []

    for file_info in answer_files:
        file_df = evaluate_file(
            dataset_name=file_info["dataset"],
            model_name=file_info["model"],
            file_name=file_info["file_name"],
            file_path=file_info["file_path"],
        )
        all_dfs.append(file_df)

    return pd.concat(all_dfs, ignore_index=True)

In [8]:
#cell8
# Build overall and per-type summaries for each separate file

def build_summary(scores_df):
    """
    Create overall and per-type macro summaries.
    Results are grouped by dataset, model, and file_name so files do not get mixed.
    """
    group_cols = ["dataset", "model", "file_name"]

    overall_df = (
        scores_df
        .groupby(group_cols, as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )
    overall_df["type"] = "overall"

    type_df = (
        scores_df
        .groupby(group_cols + ["type"], as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )

    summary_df = pd.concat([overall_df, type_df], ignore_index=True)

    summary_df["f1_percent"] = summary_df["f1_macro"] * 100

    dataset_order = ["hotpotqa", "2wikimultihopqa"]
    model_order = ["qwen3.5", "gpt-oss-120b", "gemma4"]

    summary_df["dataset"] = pd.Categorical(
        summary_df["dataset"],
        categories=dataset_order,
        ordered=True
    )

    summary_df["model"] = pd.Categorical(
        summary_df["model"],
        categories=model_order,
        ordered=True
    )

    summary_df["type_sort"] = summary_df["type"].apply(
        lambda x: "000_overall" if x == "overall" else str(x)
    )

    summary_df = (
        summary_df
        .sort_values(["dataset", "model", "file_name", "type_sort"])
        .drop(columns=["type_sort"])
        .reset_index(drop=True)
    )

    column_order = [
        "dataset",
        "model",
        "file_name",
        "type",
        "n_questions",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "f1_percent",
    ]
    summary_df = summary_df[column_order]

    numeric_cols = ["precision_macro", "recall_macro", "f1_macro", "f1_percent"]
    summary_df[numeric_cols] = summary_df[numeric_cols].round(6)

    return summary_df

In [9]:
#cell9
# Final output: separate detailed summary for every dataset-model file

scores_df = evaluate_all_files(ANSWER_FILES)
summary_df = build_summary(scores_df)

for file_name in summary_df["file_name"].unique():
    print("=" * 100)
    print(f"Results for file: {file_name}")
    print("=" * 100)

    file_summary = summary_df[summary_df["file_name"] == file_name].reset_index(drop=True)
    display(file_summary)

Results for file: hotpotqa_qwen3.5_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,hotpotqa,qwen3.5,hotpotqa_qwen3.5_answers.json,overall,1000,0.525212,0.658766,0.541898,54.189811
1,hotpotqa,qwen3.5,hotpotqa_qwen3.5_answers.json,bridge,700,0.577207,0.610074,0.571218,57.121834
2,hotpotqa,qwen3.5,hotpotqa_qwen3.5_answers.json,comparison,300,0.403889,0.772379,0.473484,47.348423


Results for file: hotpotqa_gpt_oss_120b_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,hotpotqa,gpt-oss-120b,hotpotqa_gpt_oss_120b_answers.json,overall,1000,0.542824,0.626231,0.545614,54.561387
1,hotpotqa,gpt-oss-120b,hotpotqa_gpt_oss_120b_answers.json,bridge,700,0.589364,0.597450,0.575100,57.509994
2,hotpotqa,gpt-oss-120b,hotpotqa_gpt_oss_120b_answers.json,comparison,300,0.434230,0.693387,0.476813,47.681304


Results for file: hotpotqa_gemma4_answers.json


...

,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,overall,1000,0.387812,0.510982,0.413423,41.342254
1,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,bridge_comparison,250,0.308467,0.360400,0.316613,31.661270
2,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,comparison,250,0.412295,0.790667,0.500971,50.097056
3,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,compositional,250,0.273048,0.311667,0.281086,28.108571
4,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,inference,250,0.557438,0.581195,0.555021,55.502118


Results for file: 2wikimultihopqa_gpt_oss_120b_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_gpt_oss_120b_answers.json,overall,1000,0.299876,0.385708,0.316463,31.646280
1,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_gpt_oss_120b_answers.json,bridge_comparison,250,0.028000,0.056000,0.033067,3.306667
2,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_gpt_oss_120b_answers.json,comparison,250,0.427810,0.702857,0.493202,49.320189
3,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_gpt_oss_120b_answers.json,compositional,250,0.232933,0.268200,0.236705,23.670476
4,2wikimultihopqa,gpt-oss-120b,2wikimultihopqa_gpt_oss_120b_answers.json,inference,250,0.510762,0.515776,0.502878,50.287788


Results for file: 2wikimultihopqa_gemma4_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,overall,1000,0.3329,0.397796,0.343293,34.329320
1,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,bridge_comparison,250,0.1504,0.176000,0.154667,15.466667
2,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,comparison,250,0.5646,0.732000,0.594400,59.440000
3,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,compositional,250,0.2018,0.263200,0.217061,21.706147
4,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,inference,250,0.4148,0.419986,0.407045,40.704464


In [10]:
#cell10
# Optional comparison table for easier model comparison by dataset and type

comparison_df = (
    summary_df
    .pivot_table(
        index=["dataset", "type"],
        columns="model",
        values="f1_percent",
        aggfunc="first"
    )
    .reset_index()
)

comparison_df.columns.name = None

display(comparison_df)

/tmp/ipykernel_30761/3631904528.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


,dataset,type,qwen3.5,gpt-oss-120b,gemma4
0,hotpotqa,bridge,57.121834,57.509994,51.519677
1,hotpotqa,comparison,47.348423,47.681304,53.608730
2,hotpotqa,overall,54.189811,54.561387,52.146393
3,2wikimultihopqa,bridge_comparison,31.661270,3.306667,15.466667
4,2wikimultihopqa,comparison,50.097056,49.320189,59.440000
5,2wikimultihopqa,compositional,28.108571,23.670476,21.706147
6,2wikimultihopqa,inference,55.502118,50.287788,40.704464
7,2wikimultihopqa,overall,41.342254,31.646280,34.329320
